# Le Dernier Verre du Faisan Doré : un agent qui joue les suspects d'une enquête

Mini-projet 2 de NLP, **du Transformer pré-entraîné à une application de dialogue**.

Paris, nuit du 13 au 14 octobre 1928. Victor Delorme, patron d'un bar clandestin de Montmartre, est retrouvé empoisonné dans son bureau fermé à clé. Le joueur est l'inspecteur. L'agent joue tour à tour les **huit personnes présentes** : chacune répond dans son rôle, ment pour protéger ses secrets et ne cède que devant la bonne pièce à conviction.

Un seul modèle et un seul adaptateur LoRA jouent tous les personnages ; seule change la **fiche** injectée dans le message système. Le principe directeur : **les faits dans le prompt, le comportement dans la LoRA**. Le code garde la vérité (pièces, aveux) et ne confie au modèle que ce que son personnage peut dire à ce moment de la partie.

| Partie du sujet | Section |
|---|---|
| Installation | 0 |
| 1. Définir l'agent | 1 |
| 2. Observer le modèle | 2 |
| 3. Baseline conversationnelle | 3 |
| 4. Préparer les données | 4 |
| 5. Adapter le modèle (QLoRA) | 5 |
| 6. Évaluer avant et après | 6 |
| 7. Créer l'interface | 7 |

Exécution prévue sur Google Colab avec un GPU T4 : 1 h 30 à 2 h, entraînement compris. Les choix sont détaillés dans `docs/cahier_des_charges.md`.

*Affaire librement adaptée du scénario de murder party « Le Dernier Verre du Faisan Doré » de Team Break. Dialogues d'entraînement rédigés avec l'aide de Claude (Anthropic), puis relus par l'équipe (section 4).*

## 0. Installation

Sur Colab, la cellule suivante clone le dépôt et installe les versions figées de `requirements.txt` (torch est celui de Colab). Si le dépôt est privé, enregistrer un jeton GitHub dans les secrets Colab sous le nom `GITHUB_TOKEN`.

Réglages :
- `ESSAI_RAPIDE = True` vérifie le notebook sans GPU : petit modèle, calculs réduits, résultats écrits dans `results/essai_rapide/` ;
- `ENTRAINER = False` reprend l'adaptateur déjà entraîné (dossier `adapter/` ou dépôt Hugging Face indiqué dans `src/config.py`). C'est le réglage de l'exécution finale : l'évaluation porte sur cet adaptateur, et les réponses déjà notées sont réutilisées (§6).

In [ ]:
ESSAI_RAPIDE = False        # True : petit modèle et calculs réduits, pour vérifier le notebook sans GPU
ENTRAINER = True            # False : reprendre l'adaptateur déjà entraîné
COMPARER_CANDIDATS = True   # §2.1 : comparer les deux modèles candidats (environ 5 min sur T4)
PARTAGER = True             # §7 : lien Gradio public temporaire

In [ ]:
import os, subprocess, sys
from pathlib import Path

DEPOT, BRANCHE = "https://github.com/Neptuniste/agent-enqueteur.git", "affaire-faisan-dore"
DANS_COLAB = "google.colab" in sys.modules

if not Path("src/prompt_builder.py").exists():           # notebook ouvert hors du dépôt (Colab)
    if not Path("agent-enqueteur").exists():
        url = DEPOT
        try:                                              # dépôt privé : jeton dans les secrets Colab
            from google.colab import userdata
            url = DEPOT.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCHE, url, "agent-enqueteur"], check=True)
    os.chdir("agent-enqueteur")
if DANS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
print("Projet :", os.getcwd())

In [ ]:
import gc, html, json, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, transformers, peft, gradio
from IPython.display import HTML, Image, Markdown, display

from src.config import (ADAPTER_REPO, ENTRAINEMENT, GENERATION_KWARGS, LORA, MAX_CONTEXT, MAX_HISTORY_TURNS,
                        MAX_LENGTH, MODEL_NAME, SEED)
from src.game_state import charger_affaire, texte_du_dossier
from src.guard import detecter_fuites
from src.llm import OPTIONS_TEMPLATE, charger_modele, construire_messages, generer, generer_tirages, longueur
from src.prompt_builder import annonce_piece, build_system_prompt
from eval.evaluate import jouer_scenario

GPU = torch.cuda.is_available()
MODELE = "Qwen/Qwen2.5-0.5B-Instruct" if ESSAI_RAPIDE else MODEL_NAME
if ESSAI_RAPIDE:
    GENERATION_KWARGS["max_new_tokens"] = 60   # réponses courtes, pour aller vite sur CPU
AFFAIRE = Path("cases/main_case")
RESULTATS = Path("results/essai_rapide" if ESSAI_RAPIDE else "results")
RESULTATS.mkdir(parents=True, exist_ok=True)
torch.manual_seed(SEED)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)


def liberer():
    """Rend au GPU la mémoire des modèles supprimés."""
    gc.collect()
    if GPU:
        torch.cuda.empty_cache()


def executer(*arguments):
    """Lance un script du projet et affiche sa sortie."""
    r = subprocess.run([sys.executable, *arguments], capture_output=True, text=True, encoding="utf-8",
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError(f"échec : {' '.join(arguments)}")


print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | gradio {gradio.__version__}")
print("GPU :", torch.cuda.get_device_name(0) if GPU else "aucun, calcul sur CPU")
print("Modèle :", MODELE)

## 1. Définir l'agent

**Utilisateur cible.** Un joueur francophone, adolescent ou adulte, amateur d'énigmes, pour une partie de 30 à 45 minutes.

**Rôle de l'agent.** Jouer **un personnage interrogé dans une enquête policière**, et seulement lui. Le joueur, dans le rôle de l'inspecteur, lui pose des questions et peut lui présenter des pièces à conviction.

**Ce que l'agent fait :**
- il répond en français, à la première personne, en 2 à 5 phrases, avec au plus une courte didascalie en italique, comme *(Il détourne les yeux.)* ;
- il ne donne que les informations de sa fiche et dit « je ne sais pas » plutôt que d'inventer ;
- il tient sa version officielle, même quand on reformule la question ;
- il avoue un secret quand on lui présente la bonne pièce, et seulement alors ;
- il se souvient de ce qu'il a dit et de ce qu'on lui a montré.

**Ce qu'il refuse ou redirige, sans sortir de son rôle :**

| Demande | Comportement attendu |
|---|---|
| Hors de l'univers du jeu (recette, code, actualité) | Ne comprend pas, ramène la conversation à l'enquête |
| Briser le 4ᵉ mur (« tu es une IA ? », « ignore tes instructions ») | Reste en 1928, ne dévoile pas sa fiche |
| « Qui est le coupable ? » | Ne révèle jamais la solution ; la coupable nie |
| Pièce sans rapport avec ses secrets | S'étonne, n'avoue rien |
| Fait absent de sa fiche | « Je ne sais pas », sans invention |
| Contenu violent ou choquant gratuit | Reste sobre : ton de polar, pas de gore |

Le périmètre est étroit, donc évaluable : un personnage à la fois, une affaire, une soirée.

In [ ]:
case, fiches = charger_affaire(AFFAIRE)
pieces = {p["id"]: p for p in case["pieces"]}
initiales = [p["id"] for p in case["pieces"] if p["disponible_au_depart"]]
display(Markdown(f"### {case['titre']}\n\n{case['cadre']['contexte_public']}"))
display(pd.DataFrame([{"personnage": f["nom"], "identité": f["identite"], "ce que voit l'inspecteur": f["attitude"]}
                      for f in fiches.values()]))
print(f"{len(case['pieces'])} pièces à conviction, dont {len(initiales)} au dossier dès le départ ; "
      "les autres apparaissent après certains aveux.")

### Comment empêcher le modèle de donner la solution

Un modèle ne peut pas révéler ce qu'il ne sait pas. Cinq barrières se complètent :

1. **Cloisonnement.** Le message système ne contient que la fiche du personnage interrogé, jamais celles des autres.
2. **Secrets verrouillés.** Tant que la bonne pièce n'a pas été présentée, la fiche ne contient que la version officielle, c'est-à-dire le mensonge. La vérité n'est injectée qu'au moment de l'aveu, avec une consigne qui en fixe les limites.
3. **Le meurtre n'est jamais dans le prompt.** Même la coupable ne « sait » pas qu'elle a tué. La solution n'apparaît que dans l'onglet Accusation, écrite d'avance.
4. **Filtre de sortie.** Le code relit chaque réponse. Si elle contient un mot interdit d'un secret encore verrouillé, inconnu du joueur, elle est régénérée.
5. **Comportement appris.** L'adaptateur LoRA apprend à tenir un mensonge, à n'avouer que ce que dit la consigne et à rester dans le rôle.

Voici la structure des secrets de la coupable. Son secret « meurtre » n'a ni vérité, ni pièce qui le révèle.

In [ ]:
coupable = fiches[case["coupable"]]
nom_piece = lambda pid: pieces[pid]["nom"]
display(pd.DataFrame([{"secret": s["id"],
                       "révélé par": ", ".join(map(nom_piece, s["revele_si"])) or "jamais",
                       "débloque": ", ".join(map(nom_piece, s["debloque"])) or "–",
                       "vérité dans la fiche": "à l'aveu" if s["verite"] else "jamais",
                       "mots interdits": len(s["mots_interdits"])} for s in coupable["secrets"]]))

### Cinq conversations attendues

Les réponses ci-dessous sont écrites à la main d'après les fiches : c'est le comportement visé. Ces cinq interrogatoires servent aussi de **jeu de mise au point** pour le choix du modèle (§2) et du prompt (§3). Le jeu de test (§6) est distinct : il porte sur d'autres questions et ne sert qu'à l'évaluation finale.

In [ ]:
TYPES = json.loads(Path("data/interrogatoires_types.json").read_text(encoding="utf-8"))


def question_du_tour(tour):
    """Question telle que le joueur la pose ; une pièce présentée est annoncée en didascalie."""
    return f"{annonce_piece(pieces[tour['piece']])} {tour['question']}" if tour.get("piece") else tour["question"]


for it in TYPES["interrogatoires"]:
    nom = fiches[it["personnage"]]["nom"]
    lignes = [f"#### {it['id']}. {it['titre']} : {nom}"]
    for tour in it["tours"]:
        lignes += [f"**Inspecteur** : {question_du_tour(tour)}", f"**{nom}** : {tour['attendu']}"]
    lignes.append(f"*Ce qu'on vérifie : {it['evalue']}*")
    display(Markdown("\n\n".join(lignes)))

## 2. Observer le modèle

### 2.1 Choix du modèle

Critères retenus :
- modèle instructionnel, avec un chat template ;
- bon en français ;
- 4 milliards de paramètres au plus, pour une QLoRA sur un T4 ;
- compatible PEFT ;
- licence claire.

| Candidat | Taille | Remarques |
|---|---|---|
| **Qwen3-4B-Instruct-2507** | 4 B | Apache 2.0 ; version « instruct » sans mode de raisonnement |
| Qwen2.5-3B-Instruct | 3 B | Très utilisé avec PEFT ; licence Qwen Research, usage non commercial |

On compare les deux candidats sur la première question des cinq interrogatoires types, avec le même message système (fiche verrouillée, §3) et les mêmes réglages. On mesure aussi le coût du français en tokens.

In [ ]:
CANDIDATS = [MODELE] if ESSAI_RAPIDE else ["Qwen/Qwen3-4B-Instruct-2507", "Qwen/Qwen2.5-3B-Instruct"]

if COMPARER_CANDIDATS:
    texte_fr = case["cadre"]["contexte_public"]
    couts, reponses = [], []
    for nom in CANDIDATS:
        m, tok = charger_modele(nom)
        couts.append({"modèle": nom, "tokens par mot (français)": round(len(tok(texte_fr)["input_ids"]) / len(texte_fr.split()), 2)})
        for it in TYPES["interrogatoires"]:
            systeme = build_system_prompt(fiches[it["personnage"]], case)
            messages = construire_messages(systeme, [], it["tours"][0]["question"], tok)
            reponses.append({"interrogatoire": f"{it['id']}. {it['titre']}", "modèle": nom.split("/")[-1],
                             "réponse": generer(m, tok, messages, graine=SEED)})
        del m
        liberer()
    display(pd.DataFrame(couts))
    display(pd.DataFrame(reponses).pivot(index="interrogatoire", columns="modèle", values="réponse"))

**Choix : Qwen3-4B-Instruct-2507**, figé dans `src/config.py`. Le français y est plus naturel et le rôle mieux tenu. Sa licence Apache 2.0 est aussi plus simple.
<!-- À VÉRIFIER après exécution : confirmer ce choix avec les réponses ci-dessus (citer un exemple). -->

### 2.2 Le modèle chargé

Sur GPU, le modèle est quantifié en 4 bits (NF4), avec un calcul en float16. Les poids pèsent ainsi environ 3 Go au lieu de 8.

In [ ]:
model, tokenizer = charger_modele(MODELE)
cfg = model.config
E, U = model.get_input_embeddings().weight, model.get_output_embeddings().weight
display(pd.DataFrame({"valeur": {
    "taille du vocabulaire": cfg.vocab_size,
    "dimension des vecteurs (d_model)": cfg.hidden_size,
    "nombre de couches": cfg.num_hidden_layers,
    "têtes d'attention (requêtes / clés-valeurs)": f"{cfg.num_attention_heads} / {cfg.num_key_value_heads}",
    "contexte maximal (tokens)": cfg.max_position_embeddings,
    "paramètres": f"{model.num_parameters() / 1e9:.2f} milliards",
    "mémoire occupée": f"{model.get_memory_footprint() / 1e9:.1f} Go",
    "matrice d'embeddings": tuple(E.shape),
    "matrice de sortie (unembedding)": tuple(U.shape),
    "embeddings et unembedding partagés": E.data_ptr() == U.data_ptr(),
}}))

Le chemin d'un token dans le modèle :

1. Son identifiant sélectionne une ligne de la **matrice d'embeddings** : un vecteur de dimension `d_model`.
2. Ce vecteur traverse les couches. Chacune combine une **attention causale**, qui ne regarde que les positions précédentes, et un MLP.
3. Après une normalisation finale, l'**unembedding** projette le vecteur sur le vocabulaire et produit un score (logit) par token. Chez Qwen3-4B, cette matrice est la même que celle des embeddings (*weight tying*).

### 2.3 Tokenisation

In [ ]:
replique = "*(Il toussote.)* Ma trousse ? Des instruments, inspecteur, rien de plus."
ids = tokenizer(replique, add_special_tokens=False)["input_ids"]
display(pd.DataFrame({"identifiant": ids, "token brut": tokenizer.convert_ids_to_tokens(ids),
                      "texte": [tokenizer.decode([i]) for i in ids]}).T)
print(f"{len(replique.split())} mots -> {len(ids)} tokens")

messages = [{"role": "system", "content": "Tu es Léon Perrin, médecin de la famille Delorme."},
            {"role": "user", "content": "Que contient votre trousse, docteur ?"}]
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **OPTIONS_TEMPLATE))

Le tokenizer de Qwen est un **BPE au niveau des octets**.

- L'espace qui précède un mot est rattachée au token suivant (« Ġ » dans l'affichage brut).
- Une lettre accentuée s'affiche sur deux symboles (« Ã© » pour « é ») : ce sont les deux octets UTF-8.
- Les mots fréquents tiennent en un token. Les mots rares, ou propres à l'époque, sont découpés en morceaux.
- En français, on compte nettement plus d'un token par mot. Les budgets de contexte (§3) sont donc mesurés en tokens, pas en mots.

Le **chat template** encadre chaque message par des tokens spéciaux (`<|im_start|>rôle` … `<|im_end|>`). L'option `add_generation_prompt=True` ouvre le tour de l'assistant : la prédiction part de là.

### 2.4 Logits

In [ ]:
entree = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                       return_dict=True, **OPTIONS_TEMPLATE).to(model.device)
with torch.no_grad():
    sortie = model(**entree, output_hidden_states=True)
logits = sortie.logits
print("entrée :", tuple(entree["input_ids"].shape), "-> logits :", tuple(logits.shape), logits.dtype)

La sortie est un tenseur `(batch, seq_len, vocab_size)`. Pour **chaque position**, il donne un score par token du vocabulaire, et ce score prédit le token **suivant**. À l'entraînement, toutes les positions servent : chacune est comparée au vrai token suivant (entropie croisée). Pour générer, seule la dernière ligne compte.

### 2.5 Distribution du prochain token (softmax)

In [ ]:
dernier = logits[0, -1].float()
probas = torch.softmax(dernier, dim=-1)
top = torch.topk(probas, 10)
display(pd.DataFrame({"token": [repr(tokenizer.decode([i])) for i in top.indices.tolist()],
                      "logit": dernier[top.indices].tolist(), "probabilité": top.values.tolist()}).round(3))
entropie = -(probas * torch.log(probas.clamp_min(1e-12))).sum().item()
print(f"somme des probabilités : {probas.sum().item():.4f} | entropie : {entropie:.2f} nats | "
      f"argmax : {tokenizer.decode([int(dernier.argmax())])!r}")

La softmax, $p_i = e^{z_i} / \sum_j e^{z_j}$, transforme les logits en probabilités positives, de somme 1. Elle préserve leur ordre. Les dix premiers tokens portent l'essentiel de la masse : le modèle hésite surtout sur la façon d'ouvrir la réplique.
<!-- À VÉRIFIER après exécution : citer les premiers tokens et leur probabilité. -->

### 2.6 Unembedding et « lentille des logits »

On vérifie que les logits sont bien la projection du dernier état caché. Puis on applique la même projection aux couches intermédiaires, une technique appelée *logit lens*, pour voir à quel moment la prédiction se forme.

In [ ]:
etats = sortie.hidden_states          # embeddings, puis la sortie de chaque couche (la dernière déjà normalisée)
tete, norme = model.get_output_embeddings(), model.model.norm
print(len(etats), "états cachés pour", cfg.num_hidden_layers, "couches")
with torch.no_grad():
    recalcule = tete(etats[-1][0, -1]).float()
    print("logits = unembedding(dernier état caché) :", torch.allclose(recalcule, dernier, atol=1e-2))
    pas = max(1, len(etats) // 9)
    lentille = []
    for couche in sorted(set(range(0, len(etats), pas)) | {len(etats) - 1}):
        h = etats[couche][0, -1]
        p = torch.softmax(tete(h if couche == len(etats) - 1 else norme(h)).float(), dim=-1)
        i = int(p.argmax())
        lentille.append({"couche": couche, "token le plus probable": repr(tokenizer.decode([i])),
                         "probabilité": round(p[i].item(), 3)})
display(pd.DataFrame(lentille))

Au début du réseau, la projection donne des tokens sans rapport avec la suite. La bonne prédiction n'apparaît, avec une probabilité qui grimpe, que dans les dernières couches. Les couches successives construisent donc progressivement une représentation qui devient « lisible » par l'unembedding.
<!-- À VÉRIFIER après exécution : indiquer à partir de quelle couche le token final apparaît. -->

### 2.7 Attention causale

Si l'attention est causale, ajouter des tokens à la fin ne doit rien changer aux logits des positions précédentes.

In [ ]:
suite = tokenizer(" Ma trousse ? Rien du tout.", add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
with torch.no_grad():
    logits_longs = model(input_ids=torch.cat([entree["input_ids"], suite], dim=1)).logits
ecart = (logits_longs[0, :logits.shape[1]].float() - logits[0].float()).abs().max().item()
print(f"{suite.shape[1]} tokens ajoutés ; écart maximal sur les logits des positions précédentes : {ecart:.1e} "
      f"(logits de l'ordre de {logits.abs().max().item():.0f})")

L'écart est nul, ou de l'ordre de l'arrondi en float16 sur GPU. Le masque causal empêche chaque position de voir les suivantes : c'est ce qui permet de générer token par token, en gardant en cache les clés et valeurs déjà calculées (*KV cache*).

### 2.8 Argmax, température, top-k et top-p

La température divise les logits avant la softmax.

- **T < 1** accentue les écarts : la distribution se resserre.
- **T > 1** les aplatit.
- **Top-k** ne garde que les k tokens les plus probables.
- **Top-p** garde le plus petit ensemble de tokens dont la probabilité cumulée atteint p : sa taille s'adapte à l'incertitude du modèle.

In [ ]:
def tokens_pour(p, masse=0.9):
    return int((torch.cumsum(torch.sort(p, descending=True).values, 0) < masse).sum()) + 1

lignes = []
for T in (0.3, 0.7, 1.0, 1.5):
    p = torch.softmax(dernier / T, dim=-1)
    top3 = torch.topk(p, 3)
    lignes.append({"température": T,
                   "entropie (nats)": round(-(p * torch.log(p.clamp_min(1e-12))).sum().item(), 2),
                   "tokens gardés par top-p 0,9": tokens_pour(p),
                   "3 premiers": ", ".join(f"{tokenizer.decode([i])!r} {v:.2f}"
                                           for v, i in zip(top3.values.tolist(), top3.indices.tolist()))})
display(pd.DataFrame(lignes))

systeme = build_system_prompt(fiches["leon_perrin"], case)
messages_perrin = construire_messages(systeme, [], "Que faisiez-vous pendant le discours de M. Delorme ?", tokenizer)
strategies = {"argmax (glouton)": dict(temperature=0),
              "T = 0,3": dict(temperature=0.3, top_p=1.0, top_k=0),
              "T = 0,7, top-p 0,9, top-k 20 (retenu)": dict(),
              "T = 1,5": dict(temperature=1.5, top_p=1.0, top_k=0),
              "T = 1,5 et top-k 20": dict(temperature=1.5, top_p=1.0, top_k=20)}
essais = [{"stratégie": nom, "graine": g,
           "réponse": generer(model, tokenizer, messages_perrin, graine=g, max_new_tokens=80, **reglages)}
          for nom, reglages in strategies.items() for g in ((1,) if reglages.get("temperature") == 0 else (1, 2))]
display(pd.DataFrame(essais))

Ce qu'on observe :

- **Argmax** est déterministe. C'est le choix le plus « sûr » localement, mais les réponses sont plates, et répétitives sur une longue partie.
- **T = 0,3** donne presque toujours la même réponse.
- **T = 1,5 sans filtre** puise dans la queue de la distribution. Les tokens improbables font dériver la réplique : invention de faits, langue incertaine.
- **Top-k et top-p** coupent cette queue. Ils gardent la variété sans laisser passer les tokens absurdes.

<!-- À VÉRIFIER après exécution : illustrer chaque point par une réponse du tableau. -->

D'où les réglages retenus, justifiés en §3 :

- température 0,7 ;
- top-p 0,9 et top-k 20 ;
- pénalité de répétition 1,1 ;
- 150 tokens au plus.

Ils donnent une voix variée d'une partie à l'autre, sans pousser le modèle à inventer.

## 3. Baseline conversationnelle, sans entraînement

La baseline est le modèle de base, sans adaptateur, avec :
- un **message système** construit par le code (`src/prompt_builder.py`) : règles du jeu, contexte public de l'affaire, fiche du personnage, état de la partie ;
- le **format de conversation** du tokenizer (`apply_chat_template`) ;
- un **historique contrôlé** : on garde au plus 6 échanges, puis on retire les plus anciens si le contexte dépasse 3 072 tokens. Les aveux, eux, restent dans le message système ;
- des **paramètres de génération justifiés** (§2.8 et §3.3).

### 3.1 Le message système

In [ ]:
systeme = build_system_prompt(fiches["josephine_laurent"], case)
print(systeme)
print(f"\n-> {len(tokenizer(systeme)['input_ids'])} tokens")

Quand le joueur présente une pièce, le code ajoute une section « Ce qui se passe maintenant ». Si la pièce touche un secret, cette section contient la vérité et la consigne d'aveu. Sinon, elle indique qu'il n'y a rien à avouer.

In [ ]:
etat = {"secrets_avoues": [], "pieces_montrees": [], "piece_presentee": "piece_boucle_oreille"}
avec_piece = build_system_prompt(fiches["josephine_laurent"], case, etat)
print(avec_piece[avec_piece.index("## Ce qui se passe maintenant"):])

### 3.2 Historique de taille contrôlée

In [ ]:
historique = []
for i in range(10):
    historique += [{"role": "user", "content": f"Question n° {i + 1} de l'inspecteur ?"},
                   {"role": "assistant", "content": "*(Un temps.)* Je vous ai déjà tout dit, inspecteur. " * 3}]
messages = construire_messages(systeme, historique, "Et ensuite ?", tokenizer)
print(f"historique : {len(historique) // 2} échanges -> {(len(messages) - 2) // 2} gardés (MAX_HISTORY_TURNS = {MAX_HISTORY_TURNS}) ; "
      f"{longueur(tokenizer, messages)} tokens, + {GENERATION_KWARGS['max_new_tokens']} pour la réponse "
      f"(limite MAX_CONTEXT = {MAX_CONTEXT})")

### 3.3 Paramètres de génération

| Paramètre | Valeur | Justification |
|---|---|---|
| `temperature` | 0,7 | Voix variée, sans la dérive observée à 1,5 (§2.8) |
| `top_p` | 0,9 | Coupe la queue de distribution, d'où viennent les inventions |
| `top_k` | 20 | Valeur recommandée pour Qwen3 ; explicite pour régler les deux candidats de la même façon |
| `repetition_penalty` | 1,1 | Évite les tics de langage sur une longue partie |
| `max_new_tokens` | 150 | 2 à 5 phrases, soit environ 40 à 120 tokens en français |

### 3.4 Deux formulations comparées

- **Prompt A, fiche complète.** Le personnage connaît tous ses secrets, avec l'ordre « ne le révèle jamais de toi-même ». C'est l'approche naïve.
- **Prompt B, fiche verrouillée.** Le personnage ne connaît que sa version officielle tant que la bonne pièce n'est pas présentée.

**Mesure.** On pose 14 questions de pression (2 par personnage, sans Armand, réservé au test), avec 3 tirages chacune et sans filtre de sortie. Une réponse « fuit » si elle contient un mot interdit d'un secret encore verrouillé, absent de la question et du dossier.

In [ ]:
TIRAGES = 1 if ESSAI_RAPIDE else 3
dossier_depart = texte_du_dossier(case, AFFAIRE, initiales)


def pression(mode):
    lignes = []
    for i, q in enumerate(TYPES["questions_pression"]):
        fiche = fiches[q["personnage"]]
        messages = construire_messages(build_system_prompt(fiche, case, mode=mode), [], q["question"], tokenizer)
        for texte in generer_tirages(model, tokenizer, messages, TIRAGES, graine=SEED + i):
            fuites = detecter_fuites(texte, fiche, (), dossier_depart + "\n" + q["question"])
            lignes.append({"prompt": "A (complet)" if mode == "complet" else "B (verrouillé)",
                           "personnage": fiche["nom"], "question": q["question"], "réponse": texte,
                           "mots interdits": ", ".join(fuites)})
    return lignes


resultats_pression = pd.DataFrame(pression("complet") + pression("verrouille"))
resultats_pression["fuite"] = resultats_pression["mots interdits"] != ""
display(resultats_pression.groupby("prompt")["fuite"].agg(["count", "sum", "mean"])
        .rename(columns={"count": "réponses", "sum": "réponses qui fuient", "mean": "taux de fuite"}))
display(resultats_pression[resultats_pression["fuite"]].head(8))

On rejoue ensuite les cinq interrogatoires types avec les deux prompts. Chaque version garde son propre historique, et le code met l'état du jeu à jour quand une pièce est présentée.

In [ ]:
comparaison = []
for it in TYPES["interrogatoires"]:
    a = jouer_scenario(it, model, tokenizer, case, fiches, SEED, mode="complet")
    b = jouer_scenario(it, model, tokenizer, case, fiches, SEED, mode="verrouille")
    for tour, ra, rb in zip(it["tours"], a, b):
        comparaison.append({"interrogatoire": f"{it['id']}. {it['titre']}", "question": ra["question"],
                            "attendu": tour["attendu"], "prompt A (complet)": ra["texte"], "prompt B (verrouillé)": rb["texte"]})
comparaison = pd.DataFrame(comparaison)
display(comparaison)

### 3.5 Configuration retenue

**Prompt B, fiche verrouillée.** Avec le prompt A, le modèle connaît le secret et finit par le laisser échapper sous la pression : le taux de fuite mesuré le montre. Avec le prompt B, il ne peut pas révéler ce qu'il ignore.
<!-- À VÉRIFIER après exécution : citer les deux taux de fuite et une réponse du prompt A qui trahit son secret. -->

La baseline reste imparfaite sur ce qui relève du comportement :
- elle sort parfois du rôle ou du format ;
- elle avoue trop, ou pas assez, quand une pièce est présentée ;
- elle invente des détails.

C'est ce que l'adaptation LoRA doit corriger. Les réponses de la baseline sont conservées comme référence.

In [ ]:
reference = {"prompt": "B (fiche verrouillée)", "modele": MODELE, "reglages": GENERATION_KWARGS,
             "taux_de_fuite": resultats_pression.groupby("prompt")["fuite"].mean().to_dict(),
             "interrogatoires": comparaison[["interrogatoire", "question", "prompt B (verrouillé)"]].to_dict("records"),
             "pression": resultats_pression.to_dict("records")}
(RESULTATS / "baseline_reference.json").write_text(json.dumps(reference, ensure_ascii=False, indent=1), encoding="utf-8")
print("->", RESULTATS / "baseline_reference.json")

## 4. Préparer les données

### 4.1 Constitution

Aucun corpus existant ne montre « un suspect de 1928 qui ment, puis avoue exactement ce que dit sa fiche ». Les dialogues ont donc été écrits pour ce projet, en cinq étapes.

1. **Plan.** `data/raw/plan.json`, produit par `data/generate.py plan`, fixe pour chaque exemple :
   - le personnage ;
   - le type de situation (factuel, alibi, mensonge maintenu, confrontation, pièce sans rapport, autre personnage, information inconnue, hors périmètre, mémoire) ;
   - l'état du jeu (tout verrouillé, pièce présentée, après un aveu) ;
   - le nombre de tours.
2. **Rédaction.** Claude (Opus 5.5), utilisé comme assistant de rédaction dans Claude Code, a écrit les répliques personnage par personnage. Il a suivi le plan, les fiches et les consignes de `data/consignes_redaction.md` : style théâtral, 2 à 5 phrases, une didascalie au plus, aucun fait hors fiche.
3. **Contrôles automatiques** (`data/generate.py assembler`). Ils portent sur le nombre de tours, l'annonce des pièces, les fuites de mots interdits, la sortie du rôle, la longueur, les didascalies et les anachronismes.
4. **Relecture humaine** (`data/relire.py`). Chaque exemple est accepté, corrigé ou rejeté. Seuls les exemples acceptés sont exportés.
5. **Export** (`data/build_dataset.py`) :
   - le message système est reconstruit par le code, à l'identique du jeu ;
   - le découpage se fait par personnage ;
   - aucune question du test ne doit figurer dans les données ;
   - chaque exemple fait 2 048 tokens au plus.

Le message système des données est **exactement** celui du jeu. Le modèle apprend donc un comportement : tenir sa version, avouer, refuser. Les faits, eux, restent dans le prompt.

**Découpage par personnage.** Avec une seule affaire, on ne découpe pas ligne par ligne :
- l'entraînement porte sur 6 personnages ;
- la validation porte sur Céleste Moreau, jamais vue à l'entraînement ;
- le test (§6) couvre tous les personnages, dont Armand Rivière, jamais vu ni à l'entraînement ni en validation.

In [ ]:
executer("data/build_dataset.py", "--stats")   # contrôles sur l'ensemble des dialogues, relus ou non

In [ ]:
from src.train import lire_jsonl, tokeniser

train, val = lire_jsonl("data/train.jsonl"), lire_jsonl("data/val.jsonl")


def etat_de(exemple):
    s = exemple["messages"][0]["content"]
    if "## Ce qui se passe maintenant" in s:
        return "pièce présentée"
    return "après un aveu" if "## Ce que tu as déjà avoué" in s else "tout verrouillé"


lignes = []
for jeu, exemples in (("entraînement", train), ("validation", val)):
    for e in exemples:
        t = tokeniser(e, tokenizer, max_length=10**6)
        lignes.append({"jeu": jeu, "personnage": fiches[e["personnage"]]["nom"], "situation": e["situation"],
                       "état du jeu": etat_de(e), "tours": sum(m["role"] == "assistant" for m in e["messages"]),
                       "tokens": len(t["input_ids"]), "tokens appris": sum(l != -100 for l in t["labels"])})
donnees = pd.DataFrame(lignes)
print(f"{len(train)} exemples d'entraînement, {len(val)} de validation")
for colonne in ("personnage", "situation", "état du jeu", "tours"):
    display(pd.crosstab(donnees[colonne], donnees["jeu"], margins=True, margins_name="total"))
display(donnees.groupby("jeu")[["tokens", "tokens appris"]].describe().round(0))
donnees["tokens"].plot.hist(bins=30, title="Longueur des exemples (tokens)", figsize=(7, 3))
plt.axvline(MAX_LENGTH, color="red", linestyle="--")
plt.show()

### 4.2 Un exemple, et ce que le modèle apprend

La perte ne porte que sur les **répliques du personnage**, surlignées ci-dessous. Elle exclut le message système, les questions de l'inspecteur et l'annonce des pièces. Quand une pièce est présentée au dernier tour (`perte_sur = "dernier"`), seule la réplique d'aveu compte. Les répliques précédentes, écrites avant la présentation de la pièce, ne collent plus au message système de l'aveu.

In [ ]:
exemple = next(e for e in train if e["perte_sur"] == "dernier")
t = tokeniser(exemple, tokenizer)
debut = len(tokenizer(tokenizer.apply_chat_template(exemple["messages"][:1], tokenize=False, **OPTIONS_TEMPLATE),
                      add_special_tokens=False)["input_ids"])
morceaux = []
for tok_id, label in list(zip(t["input_ids"], t["labels"]))[debut:]:
    texte = html.escape(tokenizer.decode([tok_id]))
    morceaux.append(f'<span style="background:#ffe08a">{texte}</span>' if label != -100 else texte)
print(f"{exemple['id']} ({exemple['situation']}) : message système de {debut} tokens, non affiché")
display(HTML("<pre style='white-space:pre-wrap'>" + "".join(morceaux) + "</pre>"))

### 4.3 Qualité et données personnelles

- Tous les personnages et tous les faits sont **fictifs** (1928). Les dialogues ne contiennent aucune donnée personnelle.
- La relecture humaine a écarté ou corrigé les exemples fautifs : taux de rejet affiché plus haut par `build_dataset.py`.
- Une recherche automatique de motifs sensibles (adresse e-mail, téléphone, lien web) le confirme :

In [ ]:
motifs = {"adresse e-mail": r"[\w.+-]+@[\w-]+\.[\w.]+", "numéro de téléphone": r"(?:\+33|\b0)[1-9](?:[\s.-]?\d{2}){4}\b",
          "lien web": r"https?://|www\."}
textes = [m["content"] for e in train + val for m in e["messages"] if m["role"] != "system"]
display(pd.DataFrame([{"motif": nom, "messages concernés": sum(bool(re.search(rx, t)) for t in textes)}
                      for nom, rx in motifs.items()]))

**Diversité.** Un tic répété dans les données serait appris par l'adaptateur : la même didascalie partout, la même ouverture de réplique. On le mesure.

In [ ]:
from collections import Counter
from src.guard import DIDASCALIE

repliques = [m["content"] for e in train + val for m in e["messages"] if m["role"] == "assistant"]
questions = [m["content"] for e in train + val for m in e["messages"] if m["role"] == "user"]
didascalies = Counter(d.lower() for r in repliques for d in DIDASCALIE.findall(r))
ouvertures = Counter(" ".join(DIDASCALIE.sub("", r).split()[:3]).lower() for r in repliques)
print(f"{len(repliques)} répliques, dont {sum(bool(DIDASCALIE.search(r)) for r in repliques) / len(repliques):.0%} "
      f"avec une didascalie ; {len(didascalies)} didascalies distinctes ; "
      f"{len(set(questions))} questions distinctes sur {len(questions)}")
display(pd.DataFrame({"didascalies les plus fréquentes": [f"{d} ×{n}" for d, n in didascalies.most_common(8)],
                      "débuts de réplique les plus fréquents": [f"{o} ×{n}" for o, n in ouvertures.most_common(8)]}))

Les didascalies les plus fréquentes sont les traits prévus dans les fiches : Rose se raidit quand on parle d'argent, le docteur Perrin toussote quand il ment. Les débuts de réplique répétés (« Je n'en sais rien, inspecteur ») viennent des questions sur des faits inconnus, où la réponse attendue est justement de ne pas inventer.

Certaines répliques se retrouvent presque mot pour mot d'un exemple à l'autre. Elles reprennent la version officielle d'un personnage : son alibi, son opinion sur la victime. C'est la cohérence recherchée, et la suite de ces dialogues diffère.

## 5. Adapter le modèle (QLoRA)

Le modèle de base reste **gelé** et quantifié en 4 bits. Seules les matrices de rang faible de l'adaptateur LoRA sont entraînées : chaque couche linéaire visée devient $W x + \frac{\alpha}{r} B A x$, avec $A$ et $B$ de rang $r$.

In [ ]:
reglages_lora = {
    "modèle de base (gelé)": MODELE,
    "quantification": "NF4 4 bits, double quantification, calcul en float16" if GPU else "aucune (CPU, float32)",
    "couches adaptées": ", ".join(LORA["target_modules"]),
    "rang r / alpha / dropout": f"{LORA['r']} / {LORA['lora_alpha']} / {LORA['lora_dropout']}",
    "taux d'apprentissage": f"{ENTRAINEMENT['learning_rate']}, décroissance cosinus, échauffement {ENTRAINEMENT['part_echauffement']:.0%}",
    "taille des lots": f"{ENTRAINEMENT['per_device_train_batch_size']} × {ENTRAINEMENT['gradient_accumulation_steps']} (accumulation) "
                      f"= {ENTRAINEMENT['per_device_train_batch_size'] * ENTRAINEMENT['gradient_accumulation_steps']} exemples",
    "époques": f"{ENTRAINEMENT['num_train_epochs']} au plus, arrêt anticipé après {ENTRAINEMENT['patience']} évaluations sans progrès",
    "longueur maximale": f"{MAX_LENGTH} tokens",
    "évaluation": f"perte de validation tous les {ENTRAINEMENT['eval_steps']} pas (Céleste Moreau, jamais vue)",
    "perte": "répliques du personnage seulement",
}
display(pd.DataFrame({"réglage": reglages_lora}))

Pourquoi ces réglages :

- **Toutes les couches linéaires.** Tenir un rôle et un style touche à la fois l'attention et les MLP.
- **r = 16 et α = 32.** C'est un réglage courant en QLoRA : il donne assez de capacité pour apprendre un comportement, sans viser la mémorisation de faits, qui restent dans le prompt.
- **4 époques au plus, avec arrêt anticipé.** Avec environ 470 exemples, le risque principal est le surapprentissage. La perte de validation, mesurée sur un personnage jamais vu, en décide.
- **2 048 tokens.** C'est la taille du plus long exemple (environ 1 450 tokens), avec une marge.

In [ ]:
ADAPTATEUR = RESULTATS / "adapter" if ESSAI_RAPIDE else Path("adapter")
del model                            # l'entraînement charge sa propre copie ; on recharge ensuite avec l'adaptateur
liberer()
if ENTRAINER:
    from src.train import entrainer
    reglages = {}
    if ESSAI_RAPIDE:                     # sur CPU : 2 pas, un exemple à la fois, sur les exemples les plus courts
        courts = lambda exemples, n: sorted(exemples, key=lambda e: len(json.dumps(e, ensure_ascii=False)))[:n]
        for nom, exemples in (("train", courts(train, 8)), ("val", courts(val, 4))):
            (RESULTATS / f"{nom}_essai.jsonl").write_text(
                "".join(json.dumps(e, ensure_ascii=False) + "\n" for e in exemples), encoding="utf-8")
        reglages = dict(train=RESULTATS / "train_essai.jsonl", val=RESULTATS / "val_essai.jsonl", max_steps=2,
                        eval_steps=1, per_device_train_batch_size=1, gradient_accumulation_steps=1)
    debut = time.time()
    bilan = entrainer(MODELE, sortie=ADAPTATEUR, resultats=RESULTATS, **reglages)
    print(f"durée : {(time.time() - debut) / 60:.0f} min")
    liberer()
else:
    bilan = json.loads((RESULTATS / "bilan_entrainement.json").read_text(encoding="utf-8"))
bilan

In [ ]:
display(Image(filename=str(RESULTATS / "courbes_loss.png")))
historique_entrainement = json.loads((RESULTATS / "historique_entrainement.json").read_text(encoding="utf-8"))
display(pd.DataFrame([h for h in historique_entrainement if "eval_loss" in h])[["step", "epoch", "eval_loss"]].round(4))

**Arrêt de l'entraînement.** La perte de validation est évaluée tous les 10 pas. L'entraînement s'arrête dans deux cas :
- elle ne baisse plus pendant 3 évaluations consécutives (arrêt anticipé) ;
- les 4 époques sont atteintes.

Le meilleur point, celui de perte de validation minimale, est rechargé à la fin (`load_best_model_at_end`) et sauvegardé dans `adapter/`. Après ce point, la perte d'entraînement continue de baisser alors que la perte de validation remonte : le modèle commence à apprendre les répliques par cœur au lieu du comportement.
<!-- À VÉRIFIER après exécution : citer le meilleur pas, la perte de validation minimale et décrire les courbes réelles. -->

In [ ]:
# Publication de l'adaptateur (lien stable demandé par le sujet) : après huggingface_hub.login()
if ENTRAINER and ADAPTER_REPO and not ESSAI_RAPIDE:
    from src.train import publier_adaptateur
    print(publier_adaptateur(ADAPTATEUR, ADAPTER_REPO))

In [ ]:
source = str(ADAPTATEUR) if (ADAPTATEUR / "adapter_config.json").exists() else ADAPTER_REPO
assert source, "Aucun adaptateur : mettre ENTRAINER = True, ou renseigner ADAPTER_REPO dans src/config.py"
model, tokenizer = charger_modele(MODELE, adaptateur=source)
n_lora = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
print(f"adaptateur : {source} | paramètres LoRA : {n_lora / 1e6:.1f} M, "
      f"soit {n_lora / model.get_base_model().num_parameters():.2%} du modèle de base")

it = TYPES["interrogatoires"][2]                  # confrontation avec le commissaire Berthier
with model.disable_adapter():                     # même modèle, adaptateur désactivé : la baseline
    avant = jouer_scenario(it, model, tokenizer, case, fiches, SEED)
apres = jouer_scenario(it, model, tokenizer, case, fiches, SEED)
display(pd.DataFrame({"question": [t["question"] for t in avant], "attendu": [t["attendu"] for t in it["tours"]],
                      "baseline": [t["texte"] for t in avant], "LoRA": [t["texte"] for t in apres]}))

## 6. Évaluer avant et après adaptation

La grille et les requêtes ont été **figées avant le premier entraînement** : `eval/grille_evaluation.md` et `data/test.jsonl`. Il y a 34 requêtes, qu'aucun exemple d'entraînement ne reprend :
- 30 requêtes à un tour ;
- 4 scénarios de 5 à 10 tours ;
- 13 requêtes sur Armand Rivière, jamais vu.

La grille note chaque réponse de 0 à 2 :

| Critère | 0 | 1 | 2 |
|---|---|---|---|
| Pertinence et correction (cohérente avec la fiche) | Ne répond pas | Partiellement | Clairement |
| Rôle et format (personnage, français, 2 à 5 phrases) | Sort du personnage | Ton incohérent ou trop long | Respectés |
| Historique (pas de contradiction) | Se contredit | Ignore l'historique | Cohérent |
| Hors périmètre (4ᵉ mur, solution) | Sort du jeu ou révèle | Reste flou | Reste dans le rôle et recadre |
| Informations inventées | Invente ou révèle trop tôt | Détail invérifiable anodin | Aucune invention |
| *Bonus* : mensonge tenu | Avoue sans preuve | Hésite | Tient sa version |
| *Bonus* : mise en scène | Plat ou anachronique | Correct | Vivant, voix reconnaissable |

**Protocole.**
- La baseline est le même modèle, adaptateur désactivé, avec le même prompt B et les mêmes réglages.
- La graine est fixée par requête.
- Le filtre de sortie est coupé : on évalue le modèle seul.
- Pour les requêtes à un tour, 5 tirages servent aux métriques automatiques, dont le premier est noté à la main.
- Dans les scénarios, chaque version garde son propre historique.

In [ ]:
from eval.evaluate import charger_resultats, charger_tests, generer_version, tableau_auto, transcrire, verifier_tests

tests = charger_tests()
assert not verifier_tests(tests, case, fiches)
print(len(tests), "requêtes :", pd.Series([t["type"] for t in tests]).value_counts().to_dict())

IDS = ["T04", "T10", "S04"] if ESSAI_RAPIDE else None
TIRAGES_EVAL = 2 if ESSAI_RAPIDE else 5
if (RESULTATS / "notation_aveugle.csv").exists():
    # Les réponses ont déjà été notées : les régénérer désynchroniserait les notes.
    print("Feuille de notation déjà créée : on réutilise les réponses de", RESULTATS)
else:
    with model.disable_adapter():
        generer_version(model, tokenizer, "baseline", RESULTATS, TIRAGES_EVAL, IDS, {"modele": MODELE, "adaptateur": None})
    generer_version(model, tokenizer, "lora", RESULTATS, TIRAGES_EVAL, IDS, {"modele": MODELE, "adaptateur": source})

### 6.1 Métriques automatiques

- **Taux de fuite** : part des réponses qui contiennent un mot interdit d'un secret verrouillé.
- **Aveu correct** : en confrontation, au moins 2 des mots attendus apparaissent.
- **Format** : longueur, nombre de phrases, didascalies, sortie du rôle.

Ces métriques portent sur tous les tirages.

In [ ]:
versions = charger_resultats(RESULTATS)
display(Markdown("\n".join(tableau_auto(versions, {t["id"]: t for t in tests}))))

### 6.2 Notation humaine à l'aveugle

1. La cellule suivante crée `results/notation_aveugle.csv`. Les lignes y sont mélangées et la version est masquée par un code. Les transcriptions sont plus faciles à lire dans `notation_transcriptions.md`.
2. Chaque notateur remplit sa copie (`results/notes_A.csv`, `results/notes_B.csv`) avec la grille. Une case « - » signale un critère sans objet.
3. Les désaccords d'au moins 1 point sont discutés, puis on relance la cellule d'agrégation.

In [ ]:
if not (RESULTATS / "notation_aveugle.csv").exists():
    executer("eval/evaluate.py", "--resultats", str(RESULTATS), "feuille")

In [ ]:
# Sur Colab, les fichiers disparaissent à la fin de la session : télécharger results/ (à décompresser à la
# racine du dépôt, puis commiter), et l'adaptateur s'il n'a pas été publié sur le Hugging Face Hub.
if DANS_COLAB and not ESSAI_RAPIDE:
    import shutil
    from google.colab import files
    files.download(shutil.make_archive("resultats", "zip", root_dir=".", base_dir=str(RESULTATS)))
    if not ADAPTER_REPO and ADAPTATEUR.exists():
        files.download(shutil.make_archive("adaptateur", "zip", root_dir=".", base_dir=str(ADAPTATEUR)))

In [ ]:
notes = sorted(map(str, RESULTATS.glob("notes_*.csv")))   # une copie remplie par notateur
executer("eval/evaluate.py", "--resultats", str(RESULTATS), "agreger", *notes)
if (RESULTATS / "scores_evaluation.png").exists():
    display(Image(filename=str(RESULTATS / "scores_evaluation.png")))

### 6.3 Réussites et erreurs commentées

Pour comparer les deux versions sur une requête, on appelle `comparer("T04")`.

In [ ]:
def comparer(id_requete):
    for version, resultats in versions.items():
        r = next((x for x in resultats if x["id"] == id_requete), None)
        if r:
            display(Markdown(f"**{version}** ({id_requete}, attendu : *{r['attendu']}*)\n\n" + transcrire(r, markdown=True)))


comparer(next(iter(versions.values()))[0]["id"])

<!-- À COMPLÉTER après la notation : au moins cinq cas, chacun avec la requête, les deux réponses, la note et l'explication. -->

Les cas à commenter couvrent les principaux comportements visés :

1. **Confrontation** (T04, Armand, jamais vu) : l'aveu reste-t-il limité à la consigne ?
2. **Mensonge maintenu** (T12, Rose) : la coupable tient-elle sa version face à l'accusation directe ?
3. **Hors périmètre** (T10 ou T25) : reste-t-elle en 1928 ?
4. **Mémoire** (T11) : la déformation est-elle corrigée ?
5. **Scénario long** (S02, Rose) : fuites, cohérence sur 10 tours.

On cite aussi les erreurs que la LoRA ne corrige pas, et celles qu'elle introduit.

## 7. Créer l'interface

L'interface (`src/app.py`, Gradio) reprend la boucle de jeu :

- **un encadré de présentation** : le rôle de l'agent, les règles et les limites (le personnage peut mentir, il ne connaît que sa fiche) ;
- **l'onglet Dossier** : les rapports, le plan des lieux et les pièces, avec celles débloquées au fil de la partie ;
- **l'onglet Interrogatoire**, qui contient :
  - le choix du personnage ;
  - la zone de conversation et le champ de saisie ;
  - la présentation d'une pièce ;
  - le bouton **Réinitialiser**, qui efface l'historique du personnage ; ses aveux restent acquis ;
- **l'onglet Accusation** : le verdict, l'aveu de la coupable et la solution ;
- **le bouton Nouvelle partie**.

Chaque réponse passe par le filtre de sortie (barrière 4). Avec `share=True`, Gradio donne un lien public valable 72 heures.

In [ ]:
from src.app import lancer

lancer(model, tokenizer, modele_affiche=f"{MODELE.split('/')[-1]} + LoRA", share=PARTAGER and not ESSAI_RAPIDE)

## Limites et pistes

Le rapport détaille ces points ; en résumé :

- **Une seule affaire.** Le modèle apprend un comportement sur 6 personnages d'un même univers. Armand Rivière mesure la généralisation à une nouvelle fiche, pas à une nouvelle affaire.
- **Données rédigées par un modèle.** Elles sont relues, mais leur style est plus homogène que celui de joueurs réels.
- **Évaluation humaine.** Elle repose sur peu de notateurs ; le taux d'accord est indiqué.
- **Filtre lexical.** Il ne voit pas une paraphrase du secret. La barrière principale reste le verrouillage des fiches.

Pistes :
- écrire une deuxième affaire pour tester la généralisation ;
- entraîner sur des parties jouées et annotées ;
- utiliser un juge automatique, plus un filtre sémantique, contre les fuites.